# Generalización leave-one-dataset-out

Tesis: Framework de selección inteligente de datos para el entrenamiento eficiente de modelos de lenguaje mediante aprendizaje por refuerzo y estimación del consumo energético. (Fase 4)

## Objetivo

Evaluar si una política PPO compartida puede transferirse a un dataset que no participó en su entrenamiento.

Se realizan tres particiones:

1. IMDb + AG News → TREC
2. IMDb + TREC → AG News
3. AG News + TREC → IMDb

En cada partición:

- La política PPO se reentrena desde cero usando únicamente los dos datasets de entrenamiento de la partición.
- El dataset dejado fuera no participa en el entrenamiento de PPO ni en el cálculo de su recompensa.
- El global scaler del descriptor se recalcula exclusivamente con los dos datasets fuente de la partición.
- El dataset objetivo puede proporcionar su propio conjunto de entrenamiento para construir el descriptor y realizar la selección al momento de la aplicación, pero su conjunto de validación y prueba no participan en el entrenamiento de la política.
- La evaluación final sobre el dataset dejado fuera compara Full, Random-70 y Agent mediante 5 semillas: 101, 202, 303, 404 y 505.
- La descontaminación exacta por SHA-256 se mantiene consistente con la evaluación final: el conjunto de prueba se preserva y las coincidencias exactas se eliminan de los conjuntos de entrenamiento y validación antes de construir el universo experimental.


## Protocolo congelado

In [2]:
!pip -q install \
    "stable-baselines3==2.9.0" \
    "gymnasium==1.3.0" \
    "codecarbon==3.3.1" \
    "transformers==5.16.1" \
    "datasets==4.8.5" \
    "pandas==2.2.3" \
    "numpy==2.1.3" \
    "scikit-learn==1.9.1" \
    pyarrow

In [3]:
from __future__ import annotations

from datetime import datetime, timezone
import gc
import hashlib
import importlib.metadata as md
import json
import math
import os
import random
import re
import time
import warnings
import pickle
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

import gymnasium as gym
from gymnasium import spaces
from stable_baselines3.common.vec_env import DummyVecEnv
import numpy as np
import pandas as pd
import torch
from codecarbon import EmissionsTracker
from sklearn.metrics import accuracy_score, f1_score
from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import BaseCallback
from transformers import AutoModelForSequenceClassification, AutoTokenizer

warnings.filterwarnings("ignore")

EXPERIMENT_NAME = "leave_one_dataset_out_final_v2"
SEED = 42
N_TRAIN_NOMINAL = 800
N_VAL_NOMINAL = 300
MAX_TOKEN_CAP = 256
MIN_TOKEN_CAP = 8
BATCH_SIZE = 16
TRAIN_EPOCHS = 1
LEARNING_RATE = 5e-5
WEIGHT_DECAY = 0.0
CODECARBON_MEASURE_POWER_SECS = 1
REWARD_SEEDS = [11, 22, 33]
EVAL_SEEDS = [101, 202, 303, 404, 505]
FINAL_EPISODES = 600
CHECKPOINT_EVERY_EPISODES = 12
PPO_N_STEPS = 12
PPO_BATCH_SIZE = 12
DATASETS = ["imdb", "ag_news", "trec"]
FOLDS = [
    {"source": ["imdb", "ag_news"], "target": "trec"},
    {"source": ["imdb", "trec"], "target": "ag_news"},
    {"source": ["ag_news", "trec"], "target": "imdb"},
]
DATASET_INFO = {
    "imdb": {"num_classes": 2},
    "ag_news": {"num_classes": 4},
    "trec": {"num_classes": 6},
}
DESCRIPTOR_COLUMNS = [
    "ppl_mean", "ppl_std", "ppl_p25", "ppl_p75",
    "length_mean", "length_std", "length_p25", "length_p75",
    "div_mean", "div_std", "div_p25", "div_p75",
    "num_classes_norm", "class_entropy_norm",
    "p_gt_32", "p_gt_64", "p_gt_128", "p_gt_192",
]
GLOBAL_FEATURES = ["log_perplexity", "true_length", "diversity"]

DRIVE_ROOT = Path("/content/drive/MyDrive/idsf-eft/framework")
ARTIFACT_ROOT = DRIVE_ROOT / "artifacts"
LOO_ROOT = DRIVE_ROOT / "loo_training_v2"
EVAL_ROOT = LOO_ROOT / "heldout_evaluation"
TARGET_MODEL_NAME = "distilbert/distilbert-base-uncased"

MODEL_DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
PPO_DEVICE = "cpu"
VERSIONS = {
    "stable-baselines3": md.version("stable-baselines3"),
    "gymnasium": md.version("gymnasium"),
    "codecarbon": md.version("codecarbon"),
    "transformers": md.version("transformers"),
    "datasets": md.version("datasets"),
    "pandas": md.version("pandas"),
    "numpy": md.version("numpy"),
    "scikit-learn": md.version("scikit-learn"),
    "torch": torch.__version__,
}

PPO_HYPERPARAMS = {
    "learning_rate": 3e-4,
    "n_steps": PPO_N_STEPS,
    "batch_size": PPO_BATCH_SIZE,
    "n_epochs": 10,
    "gamma": 1.0,
    "gae_lambda": 1.0,
    "clip_range": 0.2,
    "ent_coef": 0.01,
    "vf_coef": 0.5,
    "max_grad_norm": 0.5,
    "policy_net": [64, 64],
    "value_net": [64, 64],
    "device": PPO_DEVICE,
}

print("LOO root:", LOO_ROOT)
print("Target-model device:", MODEL_DEVICE)
print("PPO device:", PPO_DEVICE)
print("Checkpoint interval:", CHECKPOINT_EVERY_EPISODES, "episodes")
print("Versions:", VERSIONS)

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=

LOO root: /content/drive/MyDrive/idsf-eft/framework/loo_training_v2
Target-model device: cuda
PPO device: cpu
Checkpoint interval: 12 episodes
Versions: {'stable-baselines3': '2.9.0', 'gymnasium': '1.3.0', 'codecarbon': '3.3.1', 'transformers': '5.16.1', 'datasets': '4.8.5', 'pandas': '2.2.3', 'numpy': '2.1.3', 'scikit-learn': '1.9.1', 'torch': '2.11.0+cu130'}


In [4]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
for p in [LOO_ROOT, EVAL_ROOT]:
    p.mkdir(parents=True, exist_ok=True)
print("Drive montado y carpetas de evaluación listas.")

Drive montado y carpetas de evaluación listas.


## 1. Funciones de reproducibilidad y utilidades

In [6]:
def set_global_seed(seed: int) -> None:
    seed = int(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    try:
        torch.use_deterministic_algorithms(True, warn_only=True)
    except Exception:
        pass
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    if hasattr(torch.backends.cuda, "matmul"):
        torch.backends.cuda.matmul.allow_tf32 = False
    if hasattr(torch.backends.cudnn, "allow_tf32"):
        torch.backends.cudnn.allow_tf32 = False

def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()

def stable_hash_obj(obj: Any) -> str:
    payload = json.dumps(obj, sort_keys=True, ensure_ascii=False, default=str).encode("utf-8")
    return hashlib.sha256(payload).hexdigest()

def atomic_json_dump(obj: Any, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False, default=str)
        f.flush()
        os.fsync(f.fileno())
    os.replace(tmp, path)

def atomic_csv_dump(df: pd.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    df.to_csv(tmp, index=False)
    os.replace(tmp, path)

def atomic_sb3_save(model: PPO, dest_zip: Path) -> None:
    """Guarda un modelo SB3 de forma atómica y sin confundir la extensión .zip.
    """
    dest_zip = Path(dest_zip).with_suffix(".zip")
    dest_zip.parent.mkdir(parents=True, exist_ok=True)
    tmp_zip = dest_zip.parent / f".{dest_zip.stem}.tmp.zip"
    if tmp_zip.exists():
        tmp_zip.unlink()
    model.save(str(tmp_zip))
    if not tmp_zip.exists():
        raise RuntimeError(f"No se generó checkpoint temporal: {tmp_zip}")
    os.replace(tmp_zip, dest_zip)

def atomic_pickle_dump(obj: Any, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = Path(str(path) + ".tmp")
    with open(tmp, "wb") as f:
        pickle.dump(obj, f, protocol=pickle.HIGHEST_PROTOCOL)
        f.flush()
        os.fsync(f.fileno())
    os.replace(tmp, path)

def capture_rng_state() -> Dict[str, Any]:
    state = {
        "python_random": random.getstate(),
        "numpy_random": np.random.get_state(),
        "torch_cpu": torch.get_rng_state(),
    }
    if torch.cuda.is_available():
        state["torch_cuda"] = torch.cuda.get_rng_state_all()
    return state

def restore_rng_state(state: Dict[str, Any]) -> None:
    random.setstate(state["python_random"])
    np.random.set_state(state["numpy_random"])
    torch.set_rng_state(state["torch_cpu"])
    if torch.cuda.is_available() and "torch_cuda" in state:
        torch.cuda.set_rng_state_all(state["torch_cuda"])

def file_hash_manifest(root: Path) -> Dict[str, str]:
    out = {}
    for p in sorted(root.rglob("*")):
        if p.is_file():
            out[str(p.relative_to(root))] = sha256_file(p)
    return out

def latest_checkpoint(checkpoint_dir: Path) -> Optional[Tuple[Path, int]]:
    candidates = []
    for p in checkpoint_dir.glob("ppo_ep_*.zip"):
        m = re.fullmatch(r"ppo_ep_(\d+)\.zip", p.name)
        if m:
            candidates.append((int(m.group(1)), p))
    if not candidates:
        return None
    steps, path = max(candidates, key=lambda x: x[0])
    return path, steps

set_global_seed(SEED)

## 2. Validación de artefactos del pipeline

In [7]:
required = [
    ARTIFACT_ROOT / "pipeline_metadata.json",
]
for d in DATASETS:
    required += [
        ARTIFACT_ROOT / "datasets" / f"{d}_train.parquet",
        ARTIFACT_ROOT / "datasets" / f"{d}_validation.parquet",
        ARTIFACT_ROOT / "datasets" / f"{d}_test.parquet",
        ARTIFACT_ROOT / "features" / f"{d}_train800_features.parquet",
    ]

missing = [str(p) for p in required if not p.exists()]
if missing:
    raise FileNotFoundError("Faltan artefactos de Notebook 1:\n" + "\n".join(missing))

ARTIFACT_HASHES = file_hash_manifest(ARTIFACT_ROOT)
ARTIFACT_SIGNATURE = hashlib.sha256(
    json.dumps(ARTIFACT_HASHES, sort_keys=True).encode("utf-8")
).hexdigest()

pipeline_metadata = json.loads((ARTIFACT_ROOT / "pipeline_metadata.json").read_text(encoding="utf-8"))
assert pipeline_metadata["n_train_examples"] == 800
assert pipeline_metadata["n_val_examples"] == 300
assert pipeline_metadata["max_model_tokens"] == 256

raw_splits: Dict[str, Dict[str, pd.DataFrame]] = {}
raw_features: Dict[str, pd.DataFrame] = {}

for d in DATASETS:
    raw_features[d] = pd.read_parquet(ARTIFACT_ROOT / "features" / f"{d}_train800_features.parquet")
    raw_splits[d] = {
        "train": pd.read_parquet(ARTIFACT_ROOT / "datasets" / f"{d}_train.parquet"),
        "validation": pd.read_parquet(ARTIFACT_ROOT / "datasets" / f"{d}_validation.parquet"),
        "test": pd.read_parquet(ARTIFACT_ROOT / "datasets" / f"{d}_test.parquet"),
    }
    assert len(raw_features[d]) == 800
    assert raw_features[d]["source_index"].nunique() == 800
    assert len(raw_splits[d]["train"]) == 800
    assert len(raw_splits[d]["validation"]) == 300
    assert len(raw_splits[d]["test"]) > 0

print("Artifact signature:", ARTIFACT_SIGNATURE)
print("Artefactos cargados correctamente.")

Artifact signature: 8c4508a39bc386370f6c00f568e73ca0e2a99318ff4ab12d13afd448cd0621f1
Artefactos cargados correctamente.


## 3. Descontaminación exacta y construcción del universo de cada partición

La descontaminación se realiza antes del entrenamiento de las políticas LOO y antes de la evaluación del dataset dejado fuera. El conjunto de prueba de cada dataset se preserva íntegramente. Las coincidencias exactas se eliminan solo del conjunto de entrenamiento y validación.

In [8]:
def text_hash_series(df: pd.DataFrame) -> set[str]:
    return {
        hashlib.sha256(str(x).encode("utf-8")).hexdigest()
        for x in df["text"].astype(str).tolist()
    }

def decontaminate_dataset(
    dataset_name: str,
    split_dict: Dict[str, pd.DataFrame],
    feature_df: pd.DataFrame,
) -> Tuple[Dict[str, pd.DataFrame], pd.DataFrame, Dict[str, Any]]:
    splits = {k: v.copy() for k, v in split_dict.items()}
    feat = feature_df.copy()
    test_df = splits["test"]
    h_test = text_hash_series(test_df)

    row = {
        "dataset": dataset_name,
        "train_before": int(len(splits["train"])),
        "validation_before": int(len(splits["validation"])),
        "test_size": int(len(splits["test"])),
    }

    # Train vs Test
    train_df = splits["train"]
    h_train = text_hash_series(train_df)
    overlap = h_train & h_test
    hashes_train = train_df["text"].astype(str).map(
        lambda x: hashlib.sha256(x.encode("utf-8")).hexdigest()
    )
    mask = hashes_train.isin(overlap)
    removed = train_df.loc[mask, "source_index"].astype(int).tolist()
    row["exact_overlap_train_test"] = int(len(overlap))
    row["train_rows_removed_vs_test"] = int(mask.sum())
    row["train_removed_source_indices_vs_test"] = json.dumps(removed)
    if mask.any():
        splits["train"] = train_df.loc[~mask].reset_index(drop=True)
        feat = feat[~feat["source_index"].isin(removed)].reset_index(drop=True)

    # Validation vs Test
    val_df = splits["validation"]
    h_val = text_hash_series(val_df)
    overlap = h_val & h_test
    hashes_val = val_df["text"].astype(str).map(
        lambda x: hashlib.sha256(x.encode("utf-8")).hexdigest()
    )
    mask_val = hashes_val.isin(overlap)
    row["exact_overlap_validation_test"] = int(len(overlap))
    row["validation_rows_removed_vs_test"] = int(mask_val.sum())
    if mask_val.any():
        splits["validation"] = val_df.loc[~mask_val].reset_index(drop=True)

    # Train vs Validation
    train_df = splits["train"]
    val_df = splits["validation"]
    h_train = text_hash_series(train_df)
    h_val = text_hash_series(val_df)
    overlap = h_train & h_val
    hashes_train = train_df["text"].astype(str).map(
        lambda x: hashlib.sha256(x.encode("utf-8")).hexdigest()
    )
    mask_tv = hashes_train.isin(overlap)
    removed_tv = train_df.loc[mask_tv, "source_index"].astype(int).tolist()
    row["exact_overlap_train_validation_after_test_filter"] = int(len(overlap))
    row["train_rows_removed_vs_validation"] = int(mask_tv.sum())
    row["train_removed_source_indices_vs_validation"] = json.dumps(removed_tv)
    if mask_tv.any():
        splits["train"] = train_df.loc[~mask_tv].reset_index(drop=True)
        feat = feat[~feat["source_index"].isin(removed_tv)].reset_index(drop=True)

    row["train_after"] = int(len(splits["train"]))
    row["validation_after"] = int(len(splits["validation"]))
    row["test_after"] = int(len(splits["test"]))
    row["pdu_full_vs_nominal_train"] = float(len(feat) / N_TRAIN_NOMINAL)

    # post-filter audit
    ht = text_hash_series(splits["train"])
    hv = text_hash_series(splits["validation"])
    htest = text_hash_series(splits["test"])
    assert ht.isdisjoint(hv)
    assert ht.isdisjoint(htest)
    assert hv.isdisjoint(htest)
    assert set(feat["source_index"].astype(int)) == set(splits["train"]["source_index"].astype(int))

    return splits, feat, row

clean_splits: Dict[str, Dict[str, pd.DataFrame]] = {}
clean_features: Dict[str, pd.DataFrame] = {}
decontamination_report = []

for d in DATASETS:
    clean_splits[d], clean_features[d], row = decontaminate_dataset(
        d, raw_splits[d], raw_features[d]
    )
    decontamination_report.append(row)

# El dataset exacto de la evaluación se fija por estos mismos objetos en cada fold.
decontamination_df = pd.DataFrame(decontamination_report)
display(decontamination_df[
    ["dataset", "train_before", "train_after", "validation_before", "validation_after", "test_size",
     "exact_overlap_train_test", "exact_overlap_validation_test",
     "exact_overlap_train_validation_after_test_filter"]
])

,dataset,train_before,train_after,validation_before,validation_after,test_size,exact_overlap_train_test,exact_overlap_validation_test,exact_overlap_train_validation_after_test_filter
0,imdb,800,799,300,297,25000,1,3,0
1,ag_news,800,800,300,300,7600,0,0,0
2,trec,800,798,300,299,500,0,1,2


## 4. Normalización por ejemplo, scaler global por partición y descriptor 18D

Las variables `norm_log_perplexity`, `norm_length` y `norm_diversity` se recalculan dentro de cada dataset usando únicamente su conjunto de entrenamiento descontaminado. El global scaler usado para construir el descriptor 18D se ajusta únicamente con los dos datasets fuente de la partición.

El dataset objetivo recibe la transformación del scaler de las fuentes y nunca modifica ese scaler. De esta forma se evita incorporar estadísticas del dataset objetivo al entrenamiento de la política.

In [9]:
def robust_minmax(x: np.ndarray) -> Tuple[np.ndarray, Dict[str, float]]:
    x = np.asarray(x, dtype=float)
    lo = float(np.percentile(x, 5))
    hi = float(np.percentile(x, 95))
    if hi - lo < 1e-12:
        raise ValueError("Rango degenerado en normalización robusta.")
    z = np.clip((x - lo) / (hi - lo), 0.0, 1.0)
    return z.astype(np.float32), {"p05": lo, "p95": hi}

def rebuild_selection_norms(feature_df: pd.DataFrame) -> Tuple[pd.DataFrame, Dict[str, Any]]:
    out = feature_df.copy()
    params = {}
    for raw, norm in [
        ("log_perplexity", "norm_log_perplexity"),
        ("true_length", "norm_length"),
        ("diversity", "norm_diversity"),
    ]:
        z, p = robust_minmax(out[raw].to_numpy(float))
        out[norm] = z
        params[raw] = p
    return out, params

def fit_global_scaler(feature_tables: Dict[str, pd.DataFrame]) -> Dict[str, Dict[str, float]]:
    all_df = pd.concat([df[GLOBAL_FEATURES] for df in feature_tables.values()], ignore_index=True)
    scaler = {}
    for f in GLOBAL_FEATURES:
        x = all_df[f].to_numpy(float)
        lo = float(np.percentile(x, 5))
        hi = float(np.percentile(x, 95))
        if hi - lo < 1e-12:
            raise ValueError(f"Rango degenerado en {f}")
        scaler[f] = {"p05": lo, "p95": hi}
    return scaler

def apply_global_scaler(df: pd.DataFrame, scaler: Dict[str, Dict[str, float]]) -> pd.DataFrame:
    out = df.copy()
    for f in GLOBAL_FEATURES:
        lo = scaler[f]["p05"]
        hi = scaler[f]["p95"]
        out[f"global_norm_{f}"] = np.clip(
            (out[f].astype(float) - lo) / (hi - lo), 0.0, 1.0
        ).astype(np.float32)
    return out

def normalized_entropy(labels: pd.Series, max_classes: int = 6) -> float:
    p = labels.value_counts(normalize=True).to_numpy(float)
    H = float(-(p * np.log(p + 1e-12)).sum())
    return float(np.clip(H / np.log(max_classes), 0.0, 1.0))

def build_descriptor(feature_df: pd.DataFrame) -> pd.Series:
    ppl = feature_df["global_norm_log_perplexity"].to_numpy(float)
    L = feature_df["global_norm_true_length"].to_numpy(float)
    div = feature_df["global_norm_diversity"].to_numpy(float)
    rawL = feature_df["true_length"].to_numpy(float)
    d = {
        "ppl_mean": ppl.mean(), "ppl_std": ppl.std(ddof=1),
        "ppl_p25": np.percentile(ppl, 25), "ppl_p75": np.percentile(ppl, 75),
        "length_mean": L.mean(), "length_std": L.std(ddof=1),
        "length_p25": np.percentile(L, 25), "length_p75": np.percentile(L, 75),
        "div_mean": div.mean(), "div_std": div.std(ddof=1),
        "div_p25": np.percentile(div, 25), "div_p75": np.percentile(div, 75),
        "num_classes_norm": feature_df["label"].nunique() / 6.0,
        "class_entropy_norm": normalized_entropy(feature_df["label"], 6),
        "p_gt_32": np.mean(rawL > 32), "p_gt_64": np.mean(rawL > 64),
        "p_gt_128": np.mean(rawL > 128), "p_gt_192": np.mean(rawL > 192),
    }
    return pd.Series({k: float(np.clip(v, 0, 1)) for k, v in d.items()}, index=DESCRIPTOR_COLUMNS)

print("Funciones de normalización y descriptor listas.")

Funciones de normalización y descriptor listas.


## 5. Modelo objetivo y selección de datos

In [10]:
target_tokenizer = AutoTokenizer.from_pretrained(TARGET_MODEL_NAME)

class DynamicTextCollator:
    def __init__(self, tokenizer, max_length: int):
        self.tokenizer = tokenizer
        self.max_length = int(max_length)

    def __call__(self, batch):
        texts = [x["text"] for x in batch]
        labels = torch.tensor([int(x["label"]) for x in batch], dtype=torch.long)
        encoded = self.tokenizer(
            texts,
            padding=True,
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt",
        )
        encoded["labels"] = labels
        return encoded

def action_to_strategy(action: np.ndarray, feature_df: pd.DataFrame) -> Dict[str, float]:
    u = np.clip(np.asarray(action, dtype=np.float64), -1.0, 1.0)
    if u.shape != (5,):
        raise ValueError(f"La acción debe tener forma (5,), recibida {u.shape}")
    a = (u + 1.0) / 2.0
    raw_weights = a[:3]
    denom = float(raw_weights.sum())
    weights = np.full(3, 1.0 / 3.0) if denom <= 1e-12 else raw_weights / denom
    rho = 0.2 + 0.8 * float(a[3])
    q = 0.5 + 0.5 * float(a[4])
    lengths = feature_df["true_length"].to_numpy(float)
    q_length = float(np.quantile(lengths, q))
    max_length = int(np.clip(math.ceil(q_length), MIN_TOKEN_CAP, MAX_TOKEN_CAP))
    return {
        "rho": rho, "q": q,
        "w_ppl": float(weights[0]), "w_len": float(weights[1]), "w_div": float(weights[2]),
        "max_length": max_length,
        "u0_internal": float(u[0]), "u1_internal": float(u[1]), "u2_internal": float(u[2]),
        "u3_internal": float(u[3]), "u4_internal": float(u[4]),
    }

def score_examples(feature_df: pd.DataFrame, strategy: Dict[str, float]) -> np.ndarray:
    return (
        strategy["w_ppl"] * feature_df["norm_log_perplexity"].to_numpy(float)
        + strategy["w_len"] * (1.0 - feature_df["norm_length"].to_numpy(float))
        + strategy["w_div"] * feature_df["norm_diversity"].to_numpy(float)
    )

def stratified_top_selection(
    feature_df: pd.DataFrame,
    score: np.ndarray,
    rho: float,
    seed: int,
) -> pd.DataFrame:
    df = feature_df.copy().reset_index(drop=True)
    df["_selection_score"] = np.asarray(score, dtype=float)
    target_total = int(math.ceil(rho * len(df)))
    class_groups = {int(label): group for label, group in df.groupby("label", sort=True)}
    allocations: Dict[int, int] = {}
    residuals: Dict[int, float] = {}
    for label, group in class_groups.items():
        exact = rho * len(group)
        base = min(int(math.floor(exact)), len(group))
        allocations[label] = base
        residuals[label] = exact - base
    remaining = target_total - sum(allocations.values())
    for label in sorted(residuals, key=lambda x: (-residuals[x], x))[:max(0, remaining)]:
        allocations[label] = min(allocations[label] + 1, len(class_groups[label]))
    selected_parts = []
    for label, group in class_groups.items():
        n_select = allocations[label]
        if n_select <= 0:
            continue
        part = group.sort_values(["_selection_score", "source_index"], ascending=[False, True]).head(n_select)
        selected_parts.append(part)
    if not selected_parts:
        raise RuntimeError("La selección produjo cero ejemplos")
    selected = pd.concat(selected_parts, ignore_index=True)
    selected = selected.sample(frac=1.0, random_state=int(seed) + 1).reset_index(drop=True)
    assert len(selected) == target_total
    assert set(selected["label"].unique()) == set(df["label"].unique())
    return selected

def stratified_random_selection(feature_df: pd.DataFrame, rho: float, seed: int) -> pd.DataFrame:
    df = feature_df.copy().reset_index(drop=True)
    target_total = int(math.ceil(rho * len(df)))
    class_groups = {int(label): group for label, group in df.groupby("label", sort=True)}
    allocations, residuals = {}, {}
    for label, group in class_groups.items():
        exact = rho * len(group)
        base = min(int(math.floor(exact)), len(group))
        allocations[label] = base
        residuals[label] = exact - base
    remaining = target_total - sum(allocations.values())
    for label in sorted(residuals, key=lambda x: (-residuals[x], x))[:max(0, remaining)]:
        allocations[label] = min(allocations[label] + 1, len(class_groups[label]))
    parts = []
    for label, group in class_groups.items():
        n_select = allocations[label]
        if n_select <= 0:
            continue
        parts.append(group.sample(n=n_select, random_state=int(seed) + 1))
    selected = pd.concat(parts, ignore_index=True).sample(frac=1.0, random_state=int(seed) + 2).reset_index(drop=True)
    assert len(selected) == target_total
    assert set(selected["label"].unique()) == set(df["label"].unique())
    return selected

def apply_agent_action(feature_df: pd.DataFrame, action: np.ndarray, seed: int):
    strategy = action_to_strategy(action, feature_df)
    score = score_examples(feature_df, strategy)
    selected = stratified_top_selection(feature_df, score, strategy["rho"], seed)
    strategy["n_selected"] = int(len(selected))
    strategy["pdu_clean"] = float(len(selected) / len(feature_df))
    return selected, strategy

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

## 6. Entrenamiento y reward de PPO

El reward conserva la misma formulación de la fase 2, pero cada partición construye sus propias referencias Full usando solamente los datasets fuente.

In [11]:
def make_tracker(run_id: str, root: Path):
    run_dir = root / "codecarbon" / run_id
    run_dir.mkdir(parents=True, exist_ok=True)
    tracker = EmissionsTracker(
        project_name="tesis_framework_loo",
        experiment_id=run_id,
        measure_power_secs=CODECARBON_MEASURE_POWER_SECS,
        save_to_file=True,
        output_dir=str(run_dir),
        output_file="emissions.csv",
        save_to_api=False,
        log_level="error",
    )
    return tracker, run_dir / "emissions.csv"

def read_codecarbon_result(csv_path: Path) -> Dict[str, float]:
    if not csv_path.exists():
        raise FileNotFoundError(csv_path)
    cc = pd.read_csv(csv_path)
    if cc.empty:
        raise RuntimeError(f"CSV CodeCarbon vacío: {csv_path}")
    row = cc.iloc[-1]
    def val(name, default=np.nan):
        return float(row[name]) if name in cc.columns else float(default)
    return {
        "energy_kwh": val("energy_consumed"),
        "duration_s_codecarbon": val("duration"),
        "gpu_energy_kwh": val("gpu_energy"),
        "cpu_energy_kwh": val("cpu_energy"),
        "ram_energy_kwh": val("ram_energy"),
        "gpu_power_w": val("gpu_power"),
        "cpu_power_w": val("cpu_power"),
        "ram_power_w": val("ram_power"),
        "emissions_kg": val("emissions"),
    }

def training_signature(
    fold_name: str,
    dataset_name: str,
    seed: int,
    max_length: int,
    selected_source_indices: Optional[List[int]],
    experiment_signature: str,
) -> str:
    payload = {
        "experiment_signature": experiment_signature,
        "fold": fold_name,
        "dataset": dataset_name,
        "seed": int(seed),
        "max_length": int(max_length),
        "selected_source_indices_sha256": hashlib.sha256(
            json.dumps(selected_source_indices, sort_keys=True).encode("utf-8")
        ).hexdigest() if selected_source_indices is not None else None,
    }
    return hashlib.sha256(json.dumps(payload, sort_keys=True).encode("utf-8")).hexdigest()

def train_and_evaluate_val(
    *, fold_name: str, fold_root: Path, dataset_name: str,
    train_df: pd.DataFrame, val_df: pd.DataFrame,
    num_classes: int, max_length: int, seed: int, run_id: str,
    selected_source_indices: Optional[List[int]], experiment_signature: str,
) -> Dict[str, float]:
    run_dir = fold_root / "ppo_runs" / run_id
    result_path = run_dir / "result.json"
    meta_path = run_dir / "run_meta.json"
    run_dir.mkdir(parents=True, exist_ok=True)
    sig = training_signature(fold_name, dataset_name, seed, max_length, selected_source_indices, experiment_signature)
    if result_path.exists() and meta_path.exists():
        meta = json.loads(meta_path.read_text(encoding="utf-8"))
        if meta.get("training_signature") != sig:
            raise RuntimeError(f"Cache inconsistente: {run_id}")
        return json.loads(result_path.read_text(encoding="utf-8"))

    caller_rng_state = capture_rng_state()
    model = optimizer = train_loader = val_loader = None
    try:
        set_global_seed(seed)
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        model = AutoModelForSequenceClassification.from_pretrained(TARGET_MODEL_NAME, num_labels=num_classes)
        model.to(MODEL_DEVICE)
        optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
        train_items = train_df[["text", "label"]].to_dict("records")
        val_items = val_df[["text", "label"]].to_dict("records")
        train_loader = torch.utils.data.DataLoader(
            train_items, batch_size=BATCH_SIZE, shuffle=True,
            collate_fn=DynamicTextCollator(target_tokenizer, max_length),
            generator=torch.Generator().manual_seed(seed), num_workers=0,
        )
        val_loader = torch.utils.data.DataLoader(
            val_items, batch_size=BATCH_SIZE, shuffle=False,
            collate_fn=DynamicTextCollator(target_tokenizer, max_length), num_workers=0,
        )

        tracker, cc_csv = make_tracker(run_id, fold_root)
        t0 = time.perf_counter()
        tracker.start()
        try:
            model.train()
            for _ in range(TRAIN_EPOCHS):
                for batch in train_loader:
                    batch = {k: v.to(MODEL_DEVICE) for k, v in batch.items()}
                    optimizer.zero_grad(set_to_none=True)
                    loss = model(**batch).loss
                    loss.backward()
                    optimizer.step()
            if torch.cuda.is_available():
                torch.cuda.synchronize()
            training_time = time.perf_counter() - t0
        finally:
            tracker.stop()

        model.eval()
        y_true, y_pred = [], []
        with torch.inference_mode():
            for batch in val_loader:
                labels = batch.pop("labels")
                batch = {k: v.to(MODEL_DEVICE) for k, v in batch.items()}
                preds = torch.argmax(model(**batch).logits, dim=-1).cpu().numpy()
                y_true.extend(labels.numpy().tolist())
                y_pred.extend(preds.tolist())

        result = {
            "accuracy": float(accuracy_score(y_true, y_pred)),
            "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
            "training_time_s": float(training_time),
            **read_codecarbon_result(cc_csv),
        }
        if not np.isfinite(result["energy_kwh"]) or result["energy_kwh"] <= 0:
            raise RuntimeError(f"Energía inválida: {run_id}")
        atomic_json_dump(result, result_path)
        atomic_json_dump({"training_signature": sig, "run_id": run_id}, meta_path)
        return result
    finally:
        if model is not None:
            del model
        if optimizer is not None:
            del optimizer
        if train_loader is not None:
            del train_loader
        if val_loader is not None:
            del val_loader
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        restore_rng_state(caller_rng_state)

def compute_reward(ref: Dict[str, float], accuracy: float, macro_f1: float, energy_kwh: float) -> Tuple[float, Dict[str, float]]:
    acc_ratio = accuracy / max(ref["full_accuracy"], 1e-8)
    f1_ratio = macro_f1 / max(ref["full_macro_f1"], 1e-8)
    performance_ratio = 0.5 * (acc_ratio + f1_ratio)
    energy_saving = 1.0 - energy_kwh / max(ref["full_energy_kwh"], 1e-12)
    base_reward = 0.5 * (performance_ratio - 1.0) + 0.5 * energy_saving
    performance_penalty = 2.0 * max(0.0, 0.95 - performance_ratio)
    reward_raw = base_reward - performance_penalty
    reward = float(np.clip(reward_raw, -1.0, 1.0))
    return reward, {
        "acc_ratio": float(acc_ratio), "f1_ratio": float(f1_ratio),
        "performance_ratio": float(performance_ratio), "energy_saving": float(energy_saving),
        "base_reward": float(base_reward), "performance_penalty": float(performance_penalty),
        "reward_raw": float(reward_raw), "reward": reward,
    }

## 7. Entorno compartido de la partición

In [12]:
class LOOSharedPolicyEnvironment(gym.Env):
    metadata = {"render_modes": []}

    def __init__(self, fold_context: Dict[str, Any], episode_plan: List[str], start_episode: int = 0):
        super().__init__()
        self.ctx = fold_context
        self.episode_plan = list(episode_plan)
        self.episode_cursor = int(start_episode)
        self.current_dataset = None
        self.observation_space = spaces.Box(low=0.0, high=1.0, shape=(18,), dtype=np.float32)
        self.action_space = spaces.Box(low=-1.0, high=1.0, shape=(5,), dtype=np.float32)

    def reset(self, *, seed=None, options=None):
        super().reset(seed=seed)
        self.current_dataset = self.episode_plan[self.episode_cursor % len(self.episode_plan)]
        return self.ctx["descriptors"].loc[self.current_dataset].to_numpy(dtype=np.float32), {
            "dataset": self.current_dataset,
            "episode_index": int(self.episode_cursor),
        }

    def step(self, action):
        d = self.current_dataset
        ep = int(self.episode_cursor)
        reward_seed = REWARD_SEEDS[ep % len(REWARD_SEEDS)]
        feat = self.ctx["feature_tables"][d]
        train_df = self.ctx["splits"][d]["train"].copy()
        val_df = self.ctx["splits"][d]["validation"].copy()

        selected, strategy = apply_agent_action(feat, action, reward_seed)
        selected_idx = [int(x) for x in selected["source_index"].tolist()]
        train_df = train_df[train_df["source_index"].isin(selected_idx)].copy().sort_values("source_index").reset_index(drop=True)
        train_df["dataset"] = d

        sel_hash = hashlib.sha256(json.dumps(selected_idx, sort_keys=True).encode()).hexdigest()[:12]
        run_id = f"ppo_ep{ep:04d}_{d}_seed{reward_seed}_L{strategy['max_length']:03d}_sel{sel_hash}"
        result = train_and_evaluate_val(
            fold_name=self.ctx["fold_name"], fold_root=self.ctx["fold_root"],
            dataset_name=d, train_df=train_df, val_df=val_df,
            num_classes=DATASET_INFO[d]["num_classes"], max_length=int(strategy["max_length"]),
            seed=reward_seed, run_id=run_id, selected_source_indices=selected_idx,
            experiment_signature=self.ctx["experiment_signature"],
        )
        ref = self.ctx["full_reference"][d][str(reward_seed)]
        reward, details = compute_reward(ref, result["accuracy"], result["macro_f1"], result["energy_kwh"])
        info = {
            "dataset": d, "episode_index": ep, "reward_seed": reward_seed,
            **strategy, **result, **details,
            "selection_hash": sel_hash,
            "_selected_source_indices": selected_idx,
        }
        self.episode_cursor += 1
        next_d = self.episode_plan[self.episode_cursor % len(self.episode_plan)]
        obs = self.ctx["descriptors"].loc[next_d].to_numpy(dtype=np.float32)
        return obs, reward, True, False, info

def build_episode_plan(source_datasets: List[str], episodes: int, seed: int) -> List[str]:
    rng = np.random.RandomState(seed)
    blocks = []
    per_dataset = {d: 0 for d in source_datasets}
    for _ in range(episodes // len(source_datasets)):
        block = list(source_datasets)
        rng.shuffle(block)
        blocks.extend(block)
    if len(blocks) < episodes:
        block = list(source_datasets)
        rng.shuffle(block)
        blocks.extend(block[: episodes - len(blocks)])
    for d in blocks:
        per_dataset[d] += 1
    assert len(blocks) == episodes
    assert all(v == episodes // len(source_datasets) for v in per_dataset.values())
    return blocks

def build_ppo_model(env):
    kwargs = {
        "learning_rate": 3e-4,
        "n_steps": 12,
        "batch_size": 12,
        "n_epochs": 10,
        "gamma": 1.0,
        "gae_lambda": 1.0,
        "clip_range": 0.2,
        "ent_coef": 0.01,
        "vf_coef": 0.5,
        "max_grad_norm": 0.5,
        "policy_kwargs": {"net_arch": {"pi": [64, 64], "vf": [64, 64]}},
        "verbose": 1,
        "device": PPO_DEVICE,
    }
    return PPO("MlpPolicy", env, seed=SEED, **kwargs)

## 8. Ejecución de las tres particiones


In [13]:
def fold_name(source: List[str], target: str) -> str:
    return f"{source[0]}_{source[1]}_to_{target}"

def prepare_fold(source_datasets: List[str], target_dataset: str) -> Dict[str, Any]:
    name = fold_name(source_datasets, target_dataset)
    root = LOO_ROOT / name
    root.mkdir(parents=True, exist_ok=True)

    fold_features = {}
    selection_params = {}
    for d in DATASETS:
        fold_features[d], selection_params[d] = rebuild_selection_norms(clean_features[d])

    source_raw = {d: fold_features[d] for d in source_datasets}
    scaler = fit_global_scaler(source_raw)
    scaled = {d: apply_global_scaler(fold_features[d], scaler) for d in DATASETS}
    descriptors = pd.DataFrame({d: build_descriptor(scaled[d]) for d in DATASETS}).T
    descriptors.index.name = "dataset"
    assert descriptors.shape == (3, 18)
    assert np.isfinite(descriptors.to_numpy()).all()
    assert ((descriptors.to_numpy() >= 0) & (descriptors.to_numpy() <= 1)).all()

    fold_signature_payload = {
        "experiment_name": EXPERIMENT_NAME,
        "source_datasets": list(source_datasets),
        "target_dataset": target_dataset,
        "artifact_signature": ARTIFACT_SIGNATURE,
        "global_scaler": scaler,
        "clean_train_sizes": {d: len(clean_features[d]) for d in DATASETS},
        "clean_validation_sizes": {d: len(clean_splits[d]["validation"]) for d in DATASETS},
        "min_token_cap": MIN_TOKEN_CAP,
        "max_token_cap": MAX_TOKEN_CAP,
        "ppo_hyperparams": PPO_HYPERPARAMS,
        "reward_seeds": REWARD_SEEDS,
        "final_episodes": FINAL_EPISODES,
    }
    exp_hash = stable_hash_obj(fold_signature_payload)

    ctx = {
        "fold_name": name,
        "fold_root": root,
        "source_datasets": list(source_datasets),
        "target_dataset": target_dataset,
        "splits": {d: clean_splits[d] for d in DATASETS},
        "feature_tables": fold_features,
        "scaled_feature_tables": scaled,
        "descriptors": descriptors,
        "selection_params": selection_params,
        "global_scaler": scaler,
        "experiment_signature": exp_hash,
        "global_scaler_provenance": list(source_datasets),
    }

    atomic_json_dump(
        {
            "experiment_name": EXPERIMENT_NAME,
            "fold": name,
            "source_datasets": list(source_datasets),
            "target_dataset": target_dataset,
            "experiment_signature": exp_hash,
            "artifact_signature": ARTIFACT_SIGNATURE,
            "global_scaler_fit_on": list(source_datasets),
            "global_scaler": scaler,
            "clean_train_sizes": {d: len(clean_features[d]) for d in DATASETS},
            "clean_validation_sizes": {d: len(clean_splits[d]["validation"]) for d in DATASETS},
            "selection_normalization": "recomputed per dataset from decontaminated train",
        },
        root / "fold_setup_manifest.json",
    )
    return ctx

## 8. Referencias Full y entrenamiento LOO




In [14]:
def compute_fold_full_references(ctx: Dict[str, Any]) -> Dict[str, Dict[str, Dict[str, float]]]:
    root = ctx["fold_root"]
    ref_path = root / "full_reward_references.json"
    meta_path = root / "full_reward_references_meta.json"

    if ref_path.exists() and meta_path.exists():
        meta = json.loads(meta_path.read_text(encoding="utf-8"))
        if meta.get("experiment_signature") == ctx["experiment_signature"]:
            refs = json.loads(ref_path.read_text(encoding="utf-8"))
            if set(refs.keys()) == set(ctx["source_datasets"]):
                expected = set(map(str, REWARD_SEEDS))
                if all(set(refs[d].keys()) == expected for d in ctx["source_datasets"]):
                    print("Reutilizando referencias Full cacheadas:", ref_path)
                    return refs

    refs: Dict[str, Dict[str, Dict[str, float]]] = {}
    for d in ctx["source_datasets"]:
        refs[d] = {}
        train_df = ctx["splits"][d]["train"].copy()
        val_df = ctx["splits"][d]["validation"].copy()
        for seed in REWARD_SEEDS:
            run_id = f"ppo_full_{ctx['fold_name']}_{d}_seed{seed}_L256"
            result = train_and_evaluate_val(
                fold_name=ctx["fold_name"], fold_root=root,
                dataset_name=d, train_df=train_df, val_df=val_df,
                num_classes=DATASET_INFO[d]["num_classes"], max_length=MAX_TOKEN_CAP,
                seed=seed, run_id=run_id, selected_source_indices=None,
                experiment_signature=ctx["experiment_signature"],
            )
            refs[d][str(seed)] = {
                "full_accuracy": result["accuracy"],
                "full_macro_f1": result["macro_f1"],
                "full_energy_kwh": result["energy_kwh"],
                "full_time_s": result["training_time_s"],
                "n_clean_train": int(len(ctx["feature_tables"][d])),
            }

    atomic_json_dump(refs, ref_path)
    atomic_json_dump(
        {
            "experiment_signature": ctx["experiment_signature"],
            "source_datasets": ctx["source_datasets"],
            "reward_seeds": REWARD_SEEDS,
            "reference_sha256": sha256_file(ref_path),
        },
        meta_path,
    )
    return refs

In [15]:
class EpisodeHistoryCallback(BaseCallback):
    """Guardado de episodios"""
    def __init__(self, history_path: Path, initial_rows: Optional[List[Dict[str, Any]]] = None, verbose: int = 0):
        super().__init__(verbose)
        self.history_path = Path(history_path)
        self.rows = list(initial_rows or [])

    def _flush_history(self) -> None:
        atomic_csv_dump(pd.DataFrame(self.rows), self.history_path)

    def _on_step(self) -> bool:
        infos = self.locals.get("infos", [])
        dones = self.locals.get("dones", [])
        for info, done in zip(infos, dones):
            if bool(done) and info:
                row = {k: v for k, v in info.items() if not k.startswith("_")}
                self.rows.append(row)
                self._flush_history()
        return True

def _read_history(path: Path) -> pd.DataFrame:
    if not path.exists():
        return pd.DataFrame()
    df = pd.read_csv(path)
    if "episode_index" in df.columns:
        df = df.drop_duplicates(subset=["episode_index"], keep="last").sort_values("episode_index").reset_index(drop=True)
    return df

def checkpoint_candidates(checkpoint_dir: Path) -> List[Tuple[int, Path]]:
    out = []
    for p in checkpoint_dir.glob("ppo_ep_*.zip"):
        m = re.fullmatch(r"ppo_ep_(\d+)\.zip", p.name)
        if not m:
            continue
        steps = int(m.group(1))
        rng_path = checkpoint_dir / f"rng_ep_{steps:04d}.pkl"
        commit_path = checkpoint_dir / f"commit_ep_{steps:04d}.json"
        if not (rng_path.exists() and commit_path.exists()):
            continue
        try:
            meta = json.loads(commit_path.read_text(encoding="utf-8"))
            if int(meta.get("completed_episodes", -1)) != steps:
                continue
            if meta.get("checkpoint_sha256") != sha256_file(p):
                continue
            if meta.get("rng_sha256") != sha256_file(rng_path):
                continue
            out.append((steps, p))
        except Exception:
            continue
    return sorted(out, key=lambda x: x[0])

def latest_committed_checkpoint(checkpoint_dir: Path) -> Optional[Tuple[Path, int, Path]]:
    candidates = checkpoint_candidates(checkpoint_dir)
    if not candidates:
        return None
    steps, p = candidates[-1]
    return p, steps, checkpoint_dir / f"rng_ep_{steps:04d}.pkl"

def build_episode_plan(source_datasets: List[str], episodes: int, seed: int) -> List[str]:
    assert episodes % len(source_datasets) == 0
    rng = np.random.RandomState(seed)
    blocks = []
    per_dataset = {d: 0 for d in source_datasets}
    for _ in range(episodes // len(source_datasets)):
        block = list(source_datasets)
        rng.shuffle(block)
        blocks.extend(block)
    for d in blocks:
        per_dataset[d] += 1
    assert len(blocks) == episodes
    assert all(v == episodes // len(source_datasets) for v in per_dataset.values())
    return blocks

def build_ppo_model(env):
    return PPO(
        "MlpPolicy", env, seed=SEED,
        learning_rate=PPO_HYPERPARAMS["learning_rate"],
        n_steps=PPO_HYPERPARAMS["n_steps"],
        batch_size=PPO_HYPERPARAMS["batch_size"],
        n_epochs=PPO_HYPERPARAMS["n_epochs"],
        gamma=PPO_HYPERPARAMS["gamma"],
        gae_lambda=PPO_HYPERPARAMS["gae_lambda"],
        clip_range=PPO_HYPERPARAMS["clip_range"],
        ent_coef=PPO_HYPERPARAMS["ent_coef"],
        vf_coef=PPO_HYPERPARAMS["vf_coef"],
        max_grad_norm=PPO_HYPERPARAMS["max_grad_norm"],
        policy_kwargs={"net_arch": {"pi": PPO_HYPERPARAMS["policy_net"], "vf": PPO_HYPERPARAMS["value_net"]}},
        verbose=1,
        device=PPO_DEVICE,
    )

def train_fold_policy(ctx: Dict[str, Any]) -> Dict[str, Any]:
    root = ctx["fold_root"]
    final_model_path = root / "ppo_shared_final.zip"
    history_path = root / "ppo_training_history.csv"
    manifest_path = root / "experiment_manifest.json"
    checkpoint_dir = root / "checkpoints"
    plan_path = root / "episode_plan.json"

    ctx["full_reference"] = compute_fold_full_references(ctx)
    expected_plan = build_episode_plan(ctx["source_datasets"], FINAL_EPISODES, SEED)
    plan_signature = stable_hash_obj({"source_datasets": ctx["source_datasets"], "episodes": FINAL_EPISODES, "seed": SEED, "plan": expected_plan})
    if plan_path.exists():
        plan_meta = json.loads(plan_path.read_text(encoding="utf-8"))
        if plan_meta.get("plan_signature") != plan_signature or plan_meta.get("episode_plan") != expected_plan:
            raise RuntimeError(f"Episode plan cache inconsistente para {ctx['fold_name']}")
    else:
        atomic_json_dump({
            "plan_signature": plan_signature, "seed": SEED, "episodes": FINAL_EPISODES,
            "source_datasets": ctx["source_datasets"], "episode_plan": expected_plan,
        }, plan_path)
    episode_plan = expected_plan

    completed_cache = False
    if final_model_path.exists() and history_path.exists() and manifest_path.exists():
        manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
        cached_hist = _read_history(history_path)
        latest = latest_committed_checkpoint(checkpoint_dir)
        completed_cache = (
            manifest.get("experiment_signature") == ctx["experiment_signature"]
            and manifest.get("completed_episodes") == FINAL_EPISODES
            and len(cached_hist) == FINAL_EPISODES
            and manifest.get("checkpoint_every_episodes") == CHECKPOINT_EVERY_EPISODES
            and manifest.get("ppo_hyperparams") == PPO_HYPERPARAMS
            and latest is not None and latest[1] == FINAL_EPISODES
            and manifest.get("final_model_sha256") == sha256_file(final_model_path)
        )
        if completed_cache:
            print("Reutilizando política LOO completa cacheada:", final_model_path)
            return {"model_path": final_model_path, "history": cached_hist}

    latest = latest_committed_checkpoint(checkpoint_dir)
    history_df = _read_history(history_path)

    if latest is not None:
        checkpoint_path, completed, rng_path = latest
        if completed > FINAL_EPISODES:
            raise RuntimeError(f"Checkpoint imposible: {completed} > {FINAL_EPISODES}")
        if "episode_index" in history_df.columns:
            history_df = history_df[history_df["episode_index"] < completed].copy()
        else:
            history_df = history_df.iloc[:completed].copy()
        history_df = history_df.sort_values("episode_index").reset_index(drop=True) if "episode_index" in history_df.columns else history_df
        atomic_csv_dump(history_df, history_path)
        history_rows = history_df.to_dict("records")
        env = DummyVecEnv([lambda start=completed: LOOSharedPolicyEnvironment(ctx, episode_plan, start_episode=start)])
        model = PPO.load(str(checkpoint_path), env=env, device=PPO_DEVICE)
        with open(rng_path, "rb") as f:
            restore_rng_state(pickle.load(f))
        print(f"Reanudando {ctx['fold_name']} desde commit {completed}/{FINAL_EPISODES}.")
    else:
        history_rows = []
        env = DummyVecEnv([lambda: LOOSharedPolicyEnvironment(ctx, episode_plan, start_episode=0)])
        model = build_ppo_model(env)
        completed = 0
        print(f"Iniciando política nueva para {ctx['fold_name']} desde episodio 0.")

    while completed < FINAL_EPISODES:
        chunk = PPO_N_STEPS
        callback = EpisodeHistoryCallback(history_path=history_path, initial_rows=history_rows)
        model.learn(total_timesteps=chunk, reset_num_timesteps=False, callback=callback, progress_bar=True)
        del callback
        history_df = _read_history(history_path)
        expected_completed = completed + chunk
        if len(history_df) < expected_completed:
            raise RuntimeError(f"Historial incompleto tras actualización PPO: {len(history_df)}/{expected_completed}")
        history_df = history_df.iloc[:expected_completed].copy()
        history_rows = history_df.to_dict("records")
        completed = expected_completed

        checkpoint_dir.mkdir(parents=True, exist_ok=True)
        ckpt = checkpoint_dir / f"ppo_ep_{completed:04d}.zip"
        rng_file = checkpoint_dir / f"rng_ep_{completed:04d}.pkl"
        commit_file = checkpoint_dir / f"commit_ep_{completed:04d}.json"
        atomic_sb3_save(model, ckpt)
        atomic_pickle_dump(capture_rng_state(), rng_file)
        atomic_json_dump({
            "completed_episodes": completed,
            "checkpoint": str(ckpt),
            "checkpoint_sha256": sha256_file(ckpt),
            "rng_file": str(rng_file),
            "rng_sha256": sha256_file(rng_file),
            "history_sha256": sha256_file(history_path),
            "experiment_signature": ctx["experiment_signature"],
        }, commit_file)
        atomic_json_dump({
            "completed_episodes": completed,
            "checkpoint": str(ckpt),
            "checkpoint_sha256": sha256_file(ckpt),
            "rng_file": str(rng_file),
            "rng_sha256": sha256_file(rng_file),
            "commit_file": str(commit_file),
        }, checkpoint_dir / "latest_checkpoint.json")
        print(f"Commit PPO guardado: episodio {completed}/{FINAL_EPISODES}")

        if completed < FINAL_EPISODES:
            del env
            env = DummyVecEnv([lambda start=completed: LOOSharedPolicyEnvironment(ctx, episode_plan, start_episode=start)])
            model.set_env(env)

    atomic_sb3_save(model, final_model_path)
    hist = _read_history(history_path)
    assert len(hist) == FINAL_EPISODES
    assert hist["dataset"].isin(ctx["source_datasets"]).all()
    expected_counts = {d: FINAL_EPISODES // 2 for d in ctx["source_datasets"]}
    assert hist["dataset"].value_counts().to_dict() == expected_counts

    atomic_json_dump({
        "experiment_signature": ctx["experiment_signature"],
        "fold": ctx["fold_name"],
        "source_datasets": ctx["source_datasets"],
        "target_dataset": ctx["target_dataset"],
        "completed_episodes": FINAL_EPISODES,
        "checkpoint_every_episodes": CHECKPOINT_EVERY_EPISODES,
        "ppo_hyperparams": PPO_HYPERPARAMS,
        "final_model_sha256": sha256_file(final_model_path),
        "history_sha256": sha256_file(history_path),
        "rng_state_snapshotted": True,
        "energy_results_cached": True,
        "resumability": "commit_after_each_complete_ppo_update",
    }, manifest_path)
    del model, env
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return {"model_path": final_model_path, "history": hist}

In [16]:
fold_contexts = [prepare_fold(f["source"], f["target"]) for f in FOLDS]
for ctx in fold_contexts:
    print(" -", ctx["fold_name"], "signature:", ctx["experiment_signature"][:12])
print("Contextos de fold inicializados.")

 - imdb_ag_news_to_trec signature: ec7e4962d4d5
 - imdb_trec_to_ag_news signature: 3d639521aa45
 - ag_news_trec_to_imdb signature: fbfc2ec85c72
Contextos de fold inicializados.


In [17]:
for ctx in fold_contexts:
    print(f"=== Entrenando / Reanudando / Cargando política para: {ctx['fold_name']} ===")
    res = train_fold_policy(ctx)
    ctx["model_path"] = res["model_path"]
print("Las tres políticas LOO están entrenadas o recuperadas de caché.")

=== Entrenando / Reanudando / Cargando política para: imdb_ag_news_to_trec ===
Reutilizando referencias Full cacheadas: /content/drive/MyDrive/idsf-eft/framework/loo_training_v2/imdb_ag_news_to_trec/full_reward_references.json
Reutilizando política LOO completa cacheada: /content/drive/MyDrive/idsf-eft/framework/loo_training_v2/imdb_ag_news_to_trec/ppo_shared_final.zip
=== Entrenando / Reanudando / Cargando política para: imdb_trec_to_ag_news ===
Reutilizando referencias Full cacheadas: /content/drive/MyDrive/idsf-eft/framework/loo_training_v2/imdb_trec_to_ag_news/full_reward_references.json
Reutilizando política LOO completa cacheada: /content/drive/MyDrive/idsf-eft/framework/loo_training_v2/imdb_trec_to_ag_news/ppo_shared_final.zip
=== Entrenando / Reanudando / Cargando política para: ag_news_trec_to_imdb ===
Reutilizando referencias Full cacheadas: /content/drive/MyDrive/idsf-eft/framework/loo_training_v2/ag_news_trec_to_imdb/full_reward_references.json
Reutilizando política LOO com

## 9. Evaluación del dataset dejado fuera

Cada política se congela antes de evaluar su dataset objetivo. Se comparan Full, Random-70 y Agent con las cinco semillas finales.


In [18]:
def read_result_json(path: Path) -> Dict[str, Any]:
    return json.loads(path.read_text(encoding="utf-8"))

def train_and_evaluate_test(
    *, ctx: Dict[str, Any], condition: str, train_df: pd.DataFrame, test_df: pd.DataFrame,
    num_classes: int, max_length: int, seed: int, run_id: str,
    selected_source_indices: Optional[List[int]],
) -> Dict[str, Any]:
    root = ctx["fold_root"] / "heldout_evaluation"
    run_dir = root / "results" / run_id
    result_path = run_dir / "result.json"
    meta_path = run_dir / "run_meta.json"
    run_dir.mkdir(parents=True, exist_ok=True)
    payload = {
        "run_id": run_id,
        "fold": ctx["fold_name"], "dataset": ctx["target_dataset"], "condition": condition,
        "seed": int(seed), "max_length": int(max_length),
        "selected_source_indices_sha256": hashlib.sha256(
            json.dumps(selected_source_indices, sort_keys=True).encode()
        ).hexdigest() if selected_source_indices is not None else None,
        "policy_sha256": sha256_file(ctx["model_path"]),
    }
    sig = hashlib.sha256(json.dumps(payload, sort_keys=True).encode()).hexdigest()
    if result_path.exists() and meta_path.exists():
        meta = read_result_json(meta_path)
        if meta.get("signature") != sig:
            raise RuntimeError(f"Cache inconsistente: {run_id}")
        return read_result_json(result_path)

    set_global_seed(seed)
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    model = AutoModelForSequenceClassification.from_pretrained(TARGET_MODEL_NAME, num_labels=num_classes)
    model.to(MODEL_DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    train_items = train_df[["text", "label"]].to_dict("records")
    test_items = test_df[["text", "label"]].to_dict("records")
    train_loader = torch.utils.data.DataLoader(
        train_items, batch_size=BATCH_SIZE, shuffle=True,
        collate_fn=DynamicTextCollator(target_tokenizer, max_length),
        generator=torch.Generator().manual_seed(seed), num_workers=0,
    )
    test_loader = torch.utils.data.DataLoader(
        test_items, batch_size=BATCH_SIZE, shuffle=False,
        collate_fn=DynamicTextCollator(target_tokenizer, max_length), num_workers=0,
    )
    tracker, cc_csv = make_tracker(run_id, root)
    t0 = time.perf_counter()
    tracker.start()
    try:
        model.train()
        for _ in range(TRAIN_EPOCHS):
            for batch in train_loader:
                batch = {k: v.to(MODEL_DEVICE) for k, v in batch.items()}
                optimizer.zero_grad(set_to_none=True)
                loss = model(**batch).loss
                loss.backward()
                optimizer.step()
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        training_time = time.perf_counter() - t0
    finally:
        tracker.stop()

    model.eval()
    y_true, y_pred = [], []
    with torch.inference_mode():
        for batch in test_loader:
            labels = batch.pop("labels")
            batch = {k: v.to(MODEL_DEVICE) for k, v in batch.items()}
            preds = torch.argmax(model(**batch).logits, dim=-1).cpu().numpy()
            y_true.extend(labels.numpy().tolist())
            y_pred.extend(preds.tolist())

    cc = read_codecarbon_result(cc_csv)
    n_clean = len(ctx["feature_tables"][ctx["target_dataset"]])
    result = {
        "fold": ctx["fold_name"],
        "dataset": ctx["target_dataset"],
        "condition": condition,
        "seed": int(seed),
        "n_selected": int(len(train_df)),
        "pdu_nominal": float(len(train_df) / N_TRAIN_NOMINAL),
        "pdu_clean": float(len(train_df) / n_clean),
        "max_length": int(max_length),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "training_time_s": float(training_time),
        **cc,
    }
    if not np.isfinite(result["energy_kwh"]) or result["energy_kwh"] <= 0:
        raise RuntimeError(f"Energía inválida: {run_id}")
    atomic_json_dump(result, result_path)
    atomic_json_dump({"signature": sig, "signature_payload": payload, "run_id": run_id}, meta_path)
    del model, optimizer, train_loader, test_loader
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return result

def evaluate_heldout_fold(ctx: Dict[str, Any]) -> pd.DataFrame:
    d = ctx["target_dataset"]
    feat = ctx["feature_tables"][d].copy()
    train_full = ctx["splits"][d]["train"].copy()
    train_full["dataset"] = d
    test_df = ctx["splits"][d]["test"].copy()

    model = PPO.load(str(ctx["model_path"]), device=PPO_DEVICE)
    obs = ctx["descriptors"].loc[d].to_numpy(dtype=np.float32)
    action, _ = model.predict(obs, deterministic=True)
    action = np.asarray(action, dtype=np.float32).reshape(-1)
    selected_agent, strategy = apply_agent_action(feat, action, seed=SEED)
    agent_indices = sorted(int(x) for x in selected_agent["source_index"].tolist())
    (ctx["fold_root"] / "heldout_evaluation").mkdir(parents=True, exist_ok=True)
    pd.DataFrame([{
        "fold": ctx["fold_name"], "target_dataset": d,
        "action_0_internal": float(action[0]), "action_1_internal": float(action[1]),
        "action_2_internal": float(action[2]), "action_3_internal": float(action[3]),
        "action_4_internal": float(action[4]), **strategy,
        "n_selected": len(agent_indices),
        "pdu_nominal": len(agent_indices) / N_TRAIN_NOMINAL,
        "pdu_clean": len(agent_indices) / len(feat),
    }]).to_csv(ctx["fold_root"] / "heldout_evaluation" / "agent_action.csv", index=False)

    rows = []
    for condition in ["Full", "Random-70", "Agent"]:
        for seed in EVAL_SEEDS:
            if condition == "Full":
                selected = None
                train_subset = train_full.sort_values("source_index").reset_index(drop=True)
                max_length = MAX_TOKEN_CAP
            elif condition == "Random-70":
                selected_df = stratified_random_selection(feat, rho=0.70, seed=seed)
                selected = sorted(int(x) for x in selected_df["source_index"].tolist())
                train_subset = train_full[train_full["source_index"].isin(selected)].sort_values("source_index").reset_index(drop=True)
                max_length = MAX_TOKEN_CAP
            else:
                selected = agent_indices
                train_subset = train_full[train_full["source_index"].isin(selected)].sort_values("source_index").reset_index(drop=True)
                max_length = int(strategy["max_length"])

            sel_tag = stable_hash_obj(selected)[:12] if selected is not None else "full"
            run_id = f"eval_{ctx['fold_name']}_{condition.lower().replace('-', '')}_seed{seed}_L{max_length:03d}_{sel_tag}"
            result = train_and_evaluate_test(
                ctx=ctx, condition=condition, train_df=train_subset, test_df=test_df,
                num_classes=DATASET_INFO[d]["num_classes"], max_length=max_length, seed=seed,
                run_id=run_id, selected_source_indices=selected,
            )
            rows.append(result)
            print(ctx["fold_name"], condition, seed,
                  f"acc={result['accuracy']:.4f}", f"F1={result['macro_f1']:.4f}",
                  f"kWh={result['energy_kwh']:.7f}", f"PDUclean={result['pdu_clean']:.3f}")
    df = pd.DataFrame(rows)
    out = ctx["fold_root"] / "heldout_evaluation" / "evaluation_history.csv"
    atomic_csv_dump(df, out)
    return df

In [19]:
# Cargado/Ejecución de cada fold
all_eval = []
for ctx in fold_contexts:
    print("\n=== EVALUANDO DATASET DEJADO FUERA:", ctx["fold_name"], "===")
    df = evaluate_heldout_fold(ctx)
    all_eval.append(df)

history_df = pd.concat(all_eval, ignore_index=True)
atomic_csv_dump(history_df, EVAL_ROOT / "loo_heldout_evaluation_history.csv")
assert len(history_df) == 45
print("Total de evaluaciones LOO:", len(history_df))


=== EVALUANDO DATASET DEJADO FUERA: imdb_ag_news_to_trec ===
imdb_ag_news_to_trec Full 101 acc=0.8860 F1=0.7376 kWh=0.0000632 PDUclean=1.000
imdb_ag_news_to_trec Full 202 acc=0.8800 F1=0.7463 kWh=0.0000709 PDUclean=1.000
imdb_ag_news_to_trec Full 303 acc=0.8860 F1=0.7452 kWh=0.0000642 PDUclean=1.000
imdb_ag_news_to_trec Full 404 acc=0.8460 F1=0.7199 kWh=0.0000630 PDUclean=1.000
imdb_ag_news_to_trec Full 505 acc=0.8080 F1=0.6861 kWh=0.0000824 PDUclean=1.000
imdb_ag_news_to_trec Random-70 101 acc=0.7920 F1=0.6853 kWh=0.0000428 PDUclean=0.701
imdb_ag_news_to_trec Random-70 202 acc=0.8580 F1=0.7185 kWh=0.0000456 PDUclean=0.701
imdb_ag_news_to_trec Random-70 303 acc=0.8460 F1=0.7009 kWh=0.0000505 PDUclean=0.701
imdb_ag_news_to_trec Random-70 404 acc=0.8240 F1=0.7055 kWh=0.0000425 PDUclean=0.701
imdb_ag_news_to_trec Random-70 505 acc=0.8640 F1=0.7323 kWh=0.0000451 PDUclean=0.701
imdb_ag_news_to_trec Agent 101 acc=0.7560 F1=0.6107 kWh=0.0000308 PDUclean=0.566
imdb_ag_news_to_trec Agent 202 a

## 10. Métricas derivadas y matriz de resultados LOO


In [20]:
def add_derived_metrics(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["energy_reduction_pct"] = np.nan
    out["speedup"] = np.nan
    out["delta_accuracy"] = np.nan
    for fold in out["fold"].unique():
        mask = out["fold"] == fold
        full = out.loc[mask & (out["condition"] == "Full")].set_index("seed")
        for idx, row in out.loc[mask].iterrows():
            seed = int(row["seed"])
            out.loc[idx, "energy_reduction_pct"] = 100.0 * (1.0 - row["energy_kwh"] / full.loc[seed, "energy_kwh"])
            out.loc[idx, "speedup"] = full.loc[seed, "training_time_s"] / row["training_time_s"]
            out.loc[idx, "delta_accuracy"] = row["accuracy"] - full.loc[seed, "accuracy"]
    return out

history_df = add_derived_metrics(history_df)
atomic_csv_dump(history_df, EVAL_ROOT / "loo_heldout_evaluation_with_derived_metrics.csv")

summary = history_df.groupby(["fold", "dataset", "condition"], as_index=False).agg(
    accuracy_mean=("accuracy", "mean"), accuracy_std=("accuracy", "std"),
    macro_f1_mean=("macro_f1", "mean"), macro_f1_std=("macro_f1", "std"),
    energy_kwh_mean=("energy_kwh", "mean"), energy_kwh_std=("energy_kwh", "std"),
    training_time_s_mean=("training_time_s", "mean"), training_time_s_std=("training_time_s", "std"),
    pdu_clean_mean=("pdu_clean", "mean"), pdu_nominal_mean=("pdu_nominal", "mean"),
    energy_reduction_mean_pct=("energy_reduction_pct", "mean"),
    speedup_mean=("speedup", "mean"), delta_accuracy_mean=("delta_accuracy", "mean"),
)
atomic_csv_dump(summary, EVAL_ROOT / "loo_summary_by_fold_condition.csv")
display(summary.round(6))


,fold,dataset,condition,accuracy_mean,accuracy_std,macro_f1_mean,macro_f1_std,energy_kwh_mean,energy_kwh_std,training_time_s_mean,training_time_s_std,pdu_clean_mean,pdu_nominal_mean,energy_reduction_mean_pct,speedup_mean,delta_accuracy_mean
0,ag_news_trec_to_imdb,imdb,Agent,0.732696,0.059645,0.721633,0.072602,0.000105,0.000002,4.654401,0.039189,0.247810,0.24750,75.279699,4.116269,-0.053088
1,ag_news_trec_to_imdb,imdb,Full,0.785784,0.031706,0.780473,0.034054,0.000426,0.000006,19.157685,0.059631,1.000000,0.99875,0.000000,1.000000,0.000000
2,ag_news_trec_to_imdb,imdb,Random-70,0.800504,0.095338,0.788800,0.120446,0.000294,0.000002,13.378885,0.088579,0.700876,0.70000,31.006185,1.431997,0.014720
3,imdb_ag_news_to_trec,trec,Agent,0.690800,0.066130,0.586929,0.049614,0.000034,0.000006,1.579331,0.211335,0.566416,0.56500,49.923850,1.990623,-0.170400
4,imdb_ag_news_to_trec,trec,Full,0.861200,0.034076,0.727037,0.025176,0.000069,0.000008,3.106336,0.250606,1.000000,0.99750,0.000000,1.000000,0.000000
5,imdb_ag_news_to_trec,trec,Random-70,0.836800,0.029346,0.708487,0.017850,0.000045,0.000003,2.103462,0.090247,0.700501,0.69875,33.419453,1.480814,-0.024400
6,imdb_trec_to_ag_news,ag_news,Agent,0.868316,0.011341,0.868045,0.010948,0.000084,0.000002,3.813434,0.010258,0.852500,0.85250,41.764458,1.723730,-0.003237
7,imdb_trec_to_ag_news,ag_news,Full,0.871553,0.019167,0.871642,0.018546,0.000144,0.000004,6.573272,0.119841,1.000000,1.00000,0.000000,1.000000,0.000000
8,imdb_trec_to_ag_news,ag_news,Random-70,0.858395,0.009992,0.857631,0.010069,0.000100,0.000004,4.570373,0.123131,0.700000,0.70000,30.350799,1.439551,-0.013158


## 11. Auditoria y controles de integridad


In [21]:
assert len(history_df) == 45
assert history_df.groupby(["fold", "condition"]).size().eq(5).all()
assert sorted(history_df["seed"].unique().tolist()) == EVAL_SEEDS
assert history_df["energy_kwh"].gt(0).all()
assert history_df["training_time_s"].gt(0).all()
assert history_df["accuracy"].between(0, 1).all()
assert history_df["macro_f1"].between(0, 1).all()

for ctx in fold_contexts:
    d = ctx["target_dataset"]
    assert d not in ctx["source_datasets"]
    assert set(ctx["source_datasets"]).isdisjoint({d})
    assert ctx["source_datasets"] == list(ctx["global_scaler_provenance"])
    tr = text_hash_series(ctx["splits"][d]["train"])
    va = text_hash_series(ctx["splits"][d]["validation"])
    te = text_hash_series(ctx["splits"][d]["test"])
    assert tr.isdisjoint(va) and tr.isdisjoint(te) and va.isdisjoint(te)

for ctx in fold_contexts:
    d = ctx["target_dataset"]
    n_clean = len(ctx["feature_tables"][d])
    sub = history_df[history_df["fold"] == ctx["fold_name"]]
    full = sub[sub["condition"] == "Full"]
    rnd = sub[sub["condition"] == "Random-70"]
    agt = sub[sub["condition"] == "Agent"]
    assert (full["n_selected"] == n_clean).all()
    assert (rnd["n_selected"] == int(math.ceil(0.70 * n_clean))).all()
    assert (agt["n_selected"] > 0).all()
    assert (full["pdu_clean"] == 1.0).all()
    assert (np.isclose(rnd["pdu_clean"], int(math.ceil(0.70 * n_clean)) / n_clean)).all()

closure = pd.DataFrame([
    {"control": "45 evaluaciones LOO", "resultado": len(history_df) == 45},
    {"control": "15 evaluaciones por fold", "resultado": history_df.groupby("fold").size().eq(15).all()},
    {"control": "5 semillas por condición", "resultado": history_df.groupby(["fold", "condition"])["seed"].nunique().eq(5).all()},
    {"control": "Energía positiva", "resultado": bool(history_df["energy_kwh"].gt(0).all())},
    {"control": "Test excluido de PPO", "resultado": True},
    {"control": "Descriptor objetivo transformado con scaler de fuentes", "resultado": True},
    {"control": "Global scaler ajustado solo con fuentes", "resultado": True},
    {"control": "Train/Validation/Test textualmente disjuntos tras descontaminación", "resultado": True},
])
atomic_csv_dump(closure, EVAL_ROOT / "loo_closure_checks.csv")
display(closure)


,control,resultado
0,45 evaluaciones LOO,True
1,15 evaluaciones por fold,True
2,5 semillas por condición,True
3,Energía positiva,True
4,Test excluido de PPO,True
5,Descriptor objetivo transformado con scaler de...,True
6,Global scaler ajustado solo con fuentes,True
7,Train/Validation/Test textualmente disjuntos t...,True


## 12. Manifiesto global y archivos generados


In [22]:
manifest = {
    "experiment_name": EXPERIMENT_NAME,
    "saved_utc": datetime.now(timezone.utc).isoformat(),
    "folds": [fold_name(f["source"], f["target"]) for f in FOLDS],
    "n_folds": 3,
    "n_total_heldout_evaluations": int(len(history_df)),
    "evaluation_seeds": EVAL_SEEDS,
    "reward_seeds": REWARD_SEEDS,
    "episodes_per_fold": FINAL_EPISODES,
    "checkpoint_every_episodes": CHECKPOINT_EVERY_EPISODES,
    "target_model": TARGET_MODEL_NAME,
    "decontamination_method": "exact_text_sha256",
    "protected_split": "test",
    "filtered_splits": ["train", "validation"],
    "ppo_device": PPO_DEVICE,
    "target_model_device": str(MODEL_DEVICE),
    "global_scaler_policy": "fit only on the two source datasets in each fold",
    "target_test_used_for_ppo": False,
    "resume_capable": True,
    "resume_granularity": "12_episode_ppo_update_commit",
    "rng_state_snapshotted": True,
    "energy_results_cached": True,
    "artifact_signature": ARTIFACT_SIGNATURE,
    "loo_summary_sha256": sha256_file(EVAL_ROOT / "loo_summary_by_fold_condition.csv"),
    "history_sha256": sha256_file(EVAL_ROOT / "loo_heldout_evaluation_history.csv"),
}
atomic_json_dump(manifest, EVAL_ROOT / "loo_global_manifest.json")

print("\nArchivos principales:")
for name in [
    "loo_heldout_evaluation_history.csv",
    "loo_heldout_evaluation_with_derived_metrics.csv",
    "loo_summary_by_fold_condition.csv",
    "loo_closure_checks.csv",
    "loo_global_manifest.json",
]:
    print(EVAL_ROOT / name)
for ctx in fold_contexts:
    print(" -", ctx["fold_root"] / "ppo_shared_final.zip")



Archivos principales:
/content/drive/MyDrive/idsf-eft/framework/loo_training_v2/heldout_evaluation/loo_heldout_evaluation_history.csv
/content/drive/MyDrive/idsf-eft/framework/loo_training_v2/heldout_evaluation/loo_heldout_evaluation_with_derived_metrics.csv
/content/drive/MyDrive/idsf-eft/framework/loo_training_v2/heldout_evaluation/loo_summary_by_fold_condition.csv
/content/drive/MyDrive/idsf-eft/framework/loo_training_v2/heldout_evaluation/loo_closure_checks.csv
/content/drive/MyDrive/idsf-eft/framework/loo_training_v2/heldout_evaluation/loo_global_manifest.json
 - /content/drive/MyDrive/idsf-eft/framework/loo_training_v2/imdb_ag_news_to_trec/ppo_shared_final.zip
 - /content/drive/MyDrive/idsf-eft/framework/loo_training_v2/imdb_trec_to_ag_news/ppo_shared_final.zip
 - /content/drive/MyDrive/idsf-eft/framework/loo_training_v2/ag_news_trec_to_imdb/ppo_shared_final.zip
